# 00 - Data quality control and reproducible dataset splits

This notebook prepares the two HBB datasets for the sequential transfer-learning workflow:

1. Preserve the existing HRPlanesv2 train/validation/test split.
2. Split the official CORS-ADD-HBB `train2017` portion into approximately 85% training and 15% internal validation data.
3. Keep the official CORS-ADD-HBB `val2017` portion untouched as the held-out final test set.
4. Remove malformed or zero-area boxes only from the temporary prepared copy. The original files in Google Drive are never edited.
5. Create YOLOv5-compatible folders, YAML files, and split manifests.

Run this notebook in Colab after uploading the entire `RS_PRO` folder to Google Drive. Keep the same Colab runtime for notebook 01 because the prepared datasets are staged under `/content` for faster training.


## Expected Google Drive layout

```text
MyDrive/
└── RS_PRO/
    ├── HRPlanesv2/
    │   ├── train/
    │   ├── val/
    │   └── test/
    ├── CORS-ADD-HBB/
    │   ├── images/train2017/
    │   ├── images/val2017/
    │   ├── labels/train2017/
    │   └── labels/val2017/
    └── *.ipynb
```

The optional `CORS-ADD-OBB` folder is ignored by this workflow.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import csv
import json
import math
import os
import random
import re
import shutil
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageDraw

# Change only this line if RS_PRO is stored elsewhere in Google Drive.
PROJECT_ROOT = Path('/content/drive/MyDrive/RS_PRO')

# Prepared image data is staged on Colab local disk for faster training.
WORK_ROOT = Path('/content/rs_pro_work')
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts'

RANDOM_SEED = 42
CORS_INTERNAL_VAL_FRACTION = 0.15

# copy: uses more local disk but is usually faster during training.
# symlink: uses less local disk but reads images from Google Drive.
STAGE_MODE = 'copy'

# True safely rebuilds only /content/rs_pro_work. It never deletes Drive data.
REBUILD_PREPARED = True

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}

assert PROJECT_ROOT.exists(), f'Project folder not found: {PROJECT_ROOT}'
assert str(WORK_ROOT).startswith('/content/'), 'WORK_ROOT must remain under /content.'

if REBUILD_PREPARED and WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

WORK_ROOT.mkdir(parents=True, exist_ok=True)
(ARTIFACT_ROOT / 'splits').mkdir(parents=True, exist_ok=True)
(ARTIFACT_ROOT / 'data_qa').mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('WORK_ROOT:', WORK_ROOT)
print('STAGE_MODE:', STAGE_MODE)


## Verify the source folders


In [ ]:
required_paths = [
    PROJECT_ROOT / 'HRPlanesv2' / 'train',
    PROJECT_ROOT / 'HRPlanesv2' / 'val',
    PROJECT_ROOT / 'HRPlanesv2' / 'test',
    PROJECT_ROOT / 'CORS-ADD-HBB' / 'images' / 'train2017',
    PROJECT_ROOT / 'CORS-ADD-HBB' / 'images' / 'val2017',
    PROJECT_ROOT / 'CORS-ADD-HBB' / 'labels' / 'train2017',
    PROJECT_ROOT / 'CORS-ADD-HBB' / 'labels' / 'val2017',
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required paths:\n' + '\n'.join(missing))

for path in required_paths:
    print('OK:', path.relative_to(PROJECT_ROOT))


## Helper functions

CORS-ADD tiles are grouped by the airport/source name encoded before `_640_x_y`. The internal validation split is selected by source group rather than by individual tiles, reducing leakage from neighboring patches.


In [ ]:
def list_images(folder: Path) -> list[Path]:
    return sorted(
        path for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def cors_source_group(image_path: Path) -> str:
    match = re.match(r'^(.*)_640_\d+_\d+$', image_path.stem)
    return match.group(1) if match else image_path.stem


def select_group_validation_split(
    images: list[Path],
    fraction: float,
    seed: int,
    trials_per_group_count: int = 2000,
) -> tuple[list[Path], list[Path], set[str]]:
    groups = defaultdict(list)
    for image in images:
        groups[cors_source_group(image)].append(image)

    group_names = sorted(groups)
    target_images = round(len(images) * fraction)
    target_group_count = max(1, round(len(group_names) * fraction))
    rng = random.Random(seed)

    best_groups = None
    best_score = None

    lower = max(1, target_group_count - 3)
    upper = min(len(group_names) - 1, target_group_count + 3)

    for group_count in range(lower, upper + 1):
        for _ in range(trials_per_group_count):
            candidate = set(rng.sample(group_names, group_count))
            candidate_count = sum(len(groups[name]) for name in candidate)
            score = (
                abs(candidate_count - target_images),
                abs(group_count - target_group_count),
                sorted(candidate),
            )
            if best_score is None or score < best_score:
                best_score = score
                best_groups = candidate

    validation_images = [
        image for image in images if cors_source_group(image) in best_groups
    ]
    training_images = [
        image for image in images if cors_source_group(image) not in best_groups
    ]

    return sorted(training_images), sorted(validation_images), set(best_groups)


def stage_image(source: Path, destination: Path, mode: str) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() or destination.is_symlink():
        return

    if mode == 'copy':
        shutil.copy2(source, destination)
    elif mode == 'symlink':
        os.symlink(source, destination)
    else:
        raise ValueError("STAGE_MODE must be 'copy' or 'symlink'.")


def parse_and_clean_label(source: Path, destination: Path) -> list[dict]:
    destination.parent.mkdir(parents=True, exist_ok=True)
    retained = []
    removed = []

    if not source.exists():
        removed.append({
            'file': str(source),
            'line': None,
            'reason': 'missing_label_file',
            'content': '',
        })
        destination.write_text('', encoding='utf-8')
        return removed

    for line_number, raw_line in enumerate(
        source.read_text(encoding='utf-8', errors='replace').splitlines(), start=1
    ):
        line = raw_line.strip()
        if not line:
            continue

        parts = line.split()
        reason = None

        if len(parts) != 5:
            reason = 'expected_5_columns'
        else:
            try:
                class_id = int(float(parts[0]))
                x_center, y_center, width, height = map(float, parts[1:])
                values = [x_center, y_center, width, height]

                if class_id != 0:
                    reason = 'unexpected_class_id'
                elif not all(math.isfinite(value) for value in values):
                    reason = 'non_finite_coordinate'
                elif not (0 <= x_center <= 1 and 0 <= y_center <= 1):
                    reason = 'center_outside_normalized_range'
                elif not (0 < width <= 1 and 0 < height <= 1):
                    reason = 'zero_or_invalid_box_size'
                elif (
                    x_center - width / 2 < -1e-6
                    or x_center + width / 2 > 1 + 1e-6
                    or y_center - height / 2 < -1e-6
                    or y_center + height / 2 > 1 + 1e-6
                ):
                    reason = 'box_extends_outside_image'
            except ValueError:
                reason = 'non_numeric_label'

        if reason is None:
            retained.append(line)
        else:
            removed.append({
                'file': str(source),
                'line': line_number,
                'reason': reason,
                'content': line,
            })

    destination.write_text(
        ('\n'.join(retained) + '\n') if retained else '',
        encoding='utf-8',
    )
    return removed


def stage_yolo_split(
    items: list[tuple[Path, Path, str]],
    dataset_root: Path,
    split_name: str,
    stage_mode: str,
) -> tuple[list[dict], list[dict]]:
    manifest_rows = []
    removed_rows = []

    for image_source, label_source, source_group in items:
        image_destination = dataset_root / 'images' / split_name / image_source.name
        label_destination = dataset_root / 'labels' / split_name / f'{image_source.stem}.txt'

        stage_image(image_source, image_destination, stage_mode)
        removed_rows.extend(parse_and_clean_label(label_source, label_destination))

        manifest_rows.append({
            'split': split_name,
            'source_group': source_group,
            'source_image': str(image_source),
            'source_label': str(label_source),
            'prepared_image': str(image_destination),
            'prepared_label': str(label_destination),
        })

    return manifest_rows, removed_rows


def count_label_boxes(label_folder: Path) -> int:
    count = 0
    for label_path in label_folder.glob('*.txt'):
        count += sum(
            1 for line in label_path.read_text(encoding='utf-8').splitlines()
            if line.strip()
        )
    return count


## Prepare HRPlanesv2 without changing its official split


In [ ]:
hr_source = PROJECT_ROOT / 'HRPlanesv2'
hr_prepared = WORK_ROOT / 'hrplanesv2'

hr_manifest = []
removed_annotations = []

for split_name in ['train', 'val', 'test']:
    source_folder = hr_source / split_name
    images = list_images(source_folder)
    items = [
        (image, source_folder / f'{image.stem}.txt', f'hrplanesv2_{split_name}')
        for image in images
    ]
    manifest_rows, removed_rows = stage_yolo_split(
        items, hr_prepared, split_name, STAGE_MODE
    )
    hr_manifest.extend(manifest_rows)
    removed_annotations.extend(removed_rows)
    print(f'HRPlanesv2 {split_name}: {len(images):,} images')


## Create the group-aware CORS-ADD-HBB internal validation split


In [ ]:
cors_source = PROJECT_ROOT / 'CORS-ADD-HBB'
cors_prepared = WORK_ROOT / 'cors_add_hbb'

official_train_images = list_images(cors_source / 'images' / 'train2017')
official_test_images = list_images(cors_source / 'images' / 'val2017')

cors_train_images, cors_val_images, cors_val_groups = select_group_validation_split(
    official_train_images,
    fraction=CORS_INTERNAL_VAL_FRACTION,
    seed=RANDOM_SEED,
)

train_groups = {cors_source_group(image) for image in cors_train_images}
validation_groups = {cors_source_group(image) for image in cors_val_images}

assert train_groups.isdisjoint(validation_groups)
assert len(cors_train_images) + len(cors_val_images) == len(official_train_images)

print(f'Official train2017: {len(official_train_images):,} images')
print(f'Internal training: {len(cors_train_images):,} images')
print(f'Internal validation: {len(cors_val_images):,} images')
print(f'Validation fraction of train2017: {len(cors_val_images) / len(official_train_images):.2%}')
print(f'Held-out official val2017/test: {len(official_test_images):,} images')
print(f'Train source groups: {len(train_groups)}')
print(f'Validation source groups: {len(validation_groups)}')


In [ ]:
cors_manifest = []

cors_split_sources = {
    'train': (
        cors_train_images,
        cors_source / 'labels' / 'train2017',
    ),
    'val': (
        cors_val_images,
        cors_source / 'labels' / 'train2017',
    ),
    # The folder is named val2017 in the release, but this project reserves it
    # as a held-out final test set and never uses it for checkpoint selection.
    'test': (
        official_test_images,
        cors_source / 'labels' / 'val2017',
    ),
}

for split_name, (images, label_folder) in cors_split_sources.items():
    items = [
        (image, label_folder / f'{image.stem}.txt', cors_source_group(image))
        for image in images
    ]
    manifest_rows, removed_rows = stage_yolo_split(
        items, cors_prepared, split_name, STAGE_MODE
    )
    cors_manifest.extend(manifest_rows)
    removed_annotations.extend(removed_rows)
    print(f'CORS-ADD-HBB {split_name}: {len(images):,} images')


## Save reproducible split manifests and the data-cleaning report


In [ ]:
split_root = ARTIFACT_ROOT / 'splits'
qa_root = ARTIFACT_ROOT / 'data_qa'

hr_manifest_path = split_root / 'hrplanesv2_manifest.csv'
cors_manifest_path = split_root / 'cors_add_hbb_manifest.csv'
cors_groups_path = split_root / 'cors_internal_validation_groups.json'
removed_path = qa_root / 'removed_annotations.csv'

pd.DataFrame(hr_manifest).to_csv(hr_manifest_path, index=False)
pd.DataFrame(cors_manifest).to_csv(cors_manifest_path, index=False)
pd.DataFrame(
    removed_annotations,
    columns=['file', 'line', 'reason', 'content'],
).to_csv(removed_path, index=False)

cors_groups_path.write_text(
    json.dumps(
        {
            'random_seed': RANDOM_SEED,
            'target_validation_fraction': CORS_INTERNAL_VAL_FRACTION,
            'validation_groups': sorted(cors_val_groups),
        },
        indent=2,
    ),
    encoding='utf-8',
)

print('Saved:', hr_manifest_path)
print('Saved:', cors_manifest_path)
print('Saved:', cors_groups_path)
print('Saved:', removed_path)
print('Removed invalid annotations:', len(removed_annotations))


## Create YOLOv5 dataset YAML files


In [ ]:
config_root = WORK_ROOT / 'configs'
config_root.mkdir(parents=True, exist_ok=True)

hr_yaml = config_root / 'hrplanesv2.yaml'
cors_yaml = config_root / 'cors_add_hbb.yaml'

hr_yaml.write_text(
    f"path: {hr_prepared}\n"
    "train: images/train\n"
    "val: images/val\n"
    "test: images/test\n"
    "nc: 1\n"
    "names: ['aircraft']\n",
    encoding='utf-8',
)

cors_yaml.write_text(
    f"path: {cors_prepared}\n"
    "train: images/train\n"
    "val: images/val\n"
    "test: images/test\n"
    "nc: 1\n"
    "names: ['aircraft']\n",
    encoding='utf-8',
)

# Save archival copies in Drive. Training uses the /content versions above.
shutil.copy2(hr_yaml, ARTIFACT_ROOT / 'splits' / hr_yaml.name)
shutil.copy2(cors_yaml, ARTIFACT_ROOT / 'splits' / cors_yaml.name)

print(hr_yaml.read_text())
print(cors_yaml.read_text())


## Final prepared-data summary


In [ ]:
summary_rows = []
for dataset_name, dataset_root in [
    ('HRPlanesv2', hr_prepared),
    ('CORS-ADD-HBB', cors_prepared),
]:
    for split_name in ['train', 'val', 'test']:
        image_count = len(list_images(dataset_root / 'images' / split_name))
        box_count = count_label_boxes(dataset_root / 'labels' / split_name)
        summary_rows.append({
            'dataset': dataset_name,
            'split': split_name,
            'images': image_count,
            'valid_boxes': box_count,
        })

summary = pd.DataFrame(summary_rows)
display(summary)
summary.to_csv(ARTIFACT_ROOT / 'data_qa' / 'prepared_dataset_summary.csv', index=False)


## Visual label check


In [ ]:
def draw_yolo_boxes(image_path: Path, label_path: Path, max_boxes: int = 100) -> Image.Image:
    image = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    width, height = image.size

    lines = label_path.read_text(encoding='utf-8').splitlines()
    for line in lines[:max_boxes]:
        class_id, x_center, y_center, box_width, box_height = map(float, line.split())
        x1 = (x_center - box_width / 2) * width
        y1 = (y_center - box_height / 2) * height
        x2 = (x_center + box_width / 2) * width
        y2 = (y_center + box_height / 2) * height
        draw.rectangle((x1, y1, x2, y2), outline='red', width=max(2, width // 1000))

    return image


samples = [
    ('HRPlanesv2 train', hr_prepared / 'images' / 'train'),
    ('CORS-ADD-HBB train', cors_prepared / 'images' / 'train'),
    ('CORS-ADD-HBB internal val', cors_prepared / 'images' / 'val'),
    ('CORS-ADD-HBB held-out test', cors_prepared / 'images' / 'test'),
]

figure, axes = plt.subplots(2, 2, figsize=(14, 10))
for axis, (title, image_folder) in zip(axes.ravel(), samples):
    image_path = list_images(image_folder)[0]
    label_path = image_folder.parent.parent / 'labels' / image_folder.name / f'{image_path.stem}.txt'
    checked_image = draw_yolo_boxes(image_path, label_path)
    axis.imshow(checked_image)
    axis.set_title(title)
    axis.axis('off')

plt.tight_layout()
plt.show()


## Next step

Keep this Colab runtime active and open `01_Sequential_Transfer_Learning_YOLOv5x.ipynb`. The next notebook expects these files:

- `/content/rs_pro_work/configs/hrplanesv2.yaml`
- `/content/rs_pro_work/configs/cors_add_hbb.yaml`

If the runtime resets, rerun this notebook before resuming training.
